# Data Wrangling: Join, Combine and Reshape
## DAT540 Introduction to Data Science
## University of Stavanger
### L11.2

In [ ]:
#Importing libraries needed for pulls from Google
from pandas_datareader import data
import pandas as pd
import numpy as np
import datetime
from datetime import date

### *Concatenating Along an Axis*
- Concatenation is another form of data combination operation to concatenate, bind or stack DataFrames
- Concatenation requires the following points to be cautious off
  - Should DataFrames with different indices on axes be combined with distinct elements in these axes or only use the shared values
  - Do concatenated chunks of data need to be identified in the resulting object
  - Does "Concatenation axis" contain data that needs to be preserved 
- The *pandas.concat* top-level method provides a mechanism concat multiple pandas objects
- The concat method with a list of pandas objects stacks / concats them over axis=0 (default)
- The *join* argument specified the type of join to use. default outer
- The *join_axes* argument accepts a list of indices to be selected for joining 
- In order to identify concatenated chunks in a concatenation result, the *keys* argument can be supplied with a list of values to create hierarchical index on the concatenation axis
- In case of concatenation on DataFrame with *axis=1* argument uses the *keys* argument as DataFrame column headers
- A dict of object passed to concat instead of a list of pandas objects, the dict's keys will be automatically used as values for the keys argument
- The *names* argument allows us to name the created axis levels
- The *ignore_index=True* (default False) argument does not preserve indices along concatenation axis, instead producing a new range(total_length) index

In [ ]:
s1 = pd.Series([-1, 1], index=['a', 'b'])
s2 = pd.Series([2, -1, 4], index=['c', 'a', 'f'])
s3 = pd.Series([5, -3], index=['f', 'a'])

In [ ]:
pd.concat([s1, s2, s3])

In [ ]:
pd.concat([s1, s2, s3], sort=True)

In [ ]:
# Creating intersection using join type as inner
pd.concat([s1, s2, s3], axis=1, sort=True)

In [ ]:
# Creating a hierarchical index for each concatenated piece
pd.concat([s1, s2, s3], keys=['c1', 'c2', 'c3'])

In [ ]:
# Concatenating DataFrames
df1 = pd.DataFrame(np.arange(6).reshape(3,2), index=['a', 'b', 'c'], columns=['c1', 'c2'])
df2 = pd.DataFrame(5 + np.arange(4).reshape(2,2), index=['a', 'c'], columns=['c3', 'c4'])

In [ ]:
print(df1)
print(df2)

In [ ]:
pd.concat([df1, df2], axis=1, keys=['df1', 'df2'], sort=True, join="inner")# use keys to identify concatenated pieces

In [ ]:
# pass a dict of objects instead of a list , keys arguments will be the dict keys
pd.concat({'df1': df1, 'df2': df2}, axis=1, sort=True)

- *Combining Data with Overlap*
- In cases where datasets have indices that overlap in full or in part. We can choose values from either one when one of them has null values
- In other words, how do we patch missing data in one object from another
- Similar to ```numpy.where(pd.isnull(a), b, a)```
- Series has a *combine_first* instance method for performing equivalent operations
- With DataFrames, *combine_first* does the same thing column by column

In [ ]:
a = pd.Series([np.nan, 2.5, 0.0, 3.5, 4.5, np.nan])
b = pd.Series([0.0, np.nan, 2.0, np.nan, np.nan, 5.])

In [ ]:
print(a)
print(b)

In [ ]:
# We can combine two objects based on any conditions
b.combine(a, lambda bi, ai: ai if pd.isnull(bi) else bi)

In [ ]:
# The above can be also achieved using combine_first
b.combine_first(a)

In [ ]:
# combune first can be used on DataFrames as well; it does the same thing column by column
df1 = pd.DataFrame({'c1': [0., np.nan, 2., np.nan],
                    'c2': [np.nan, 3., np.nan, 4.]},
                   index=['a', 'b', 'c', 'd'])
df2 = pd.DataFrame({'c1': [5., 6., 7., 8.],
                    'c2': [9., 10., 11., 12.]},
                   index=['a', 'b', 'c', 'd'])  
print(df1)
print(df2)

In [ ]:
df1.combine_first(df2)

- **Reshaping and Pivoting**
- Allows rearrangement of tabular data. Also referred to as *reshape* or *pivot* operations
- *Reshaping with Hierarchical Indexing*
- Hierarchical indexing provides a consistent way to rearrange data in a DataFrame
- It provides two primary actions through instance methods:
  - *stack*: rotates or pivots columns to rows
  - *unstack*: transforms rows into the columns
- by default the innermost level is stacked or unstacked. A different level can be chosen by passing a level number or name
- Unlike stacking that filters out missing data by default (*dropna=True*), unstacking might introduce missing data if all the values in the level aren't found in each subgroups
- When unstacking a DataFrame, the level unstacked becomes the lowest level in the result

In [ ]:
#Let us have a simple DataFrame
sdf = pd.DataFrame(np.arange(9).reshape(3,3), index=list('abc'), columns=list('xyz'))
sdf

In [ ]:
# Stacking the data would pivot the columns into rows producing a Series
s1 = sdf.stack()
print('s1 type: ', type(s1))
s1

In [ ]:
# UnStacking transforms the Series by pivoting the rows into columns. By default, it takes the inner most level (x,y,z)
s1.unstack()

In [ ]:
# This can be changed, by providing the level argument with the level that we want to pivot by
s1.unstack(0)

- *Pivoting "Wide" to "Long" Format*
- It merges multiple columns into one, producing a DataFrame that is longer than the input
- *pandas.melt* top-level method allows us to perform this transformation.
- It accepts a DataFrame and a list of column names to the *id_vars* argument on which to convert to a long format
- a *value_vars* argument accepts a list and selects the columns that would be represented as values
- *pandas.melt* can also be used without any group identifiers to create the long format without any labeled indices


In [ ]:
start_date = datetime.datetime(2021, 1, 1)
end_date = datetime.datetime(2021, 9, 30)
aapl = data.DataReader(name='aapl', data_source='stooq', start=start_date, end=end_date)[['Close']]
# get qqq stocks and store it to an instance
qqq = data.DataReader(name='qqq', data_source='stooq', start=start_date, end=end_date)[['Close']]
# get ^DJI stocks and store it to an instance
bbby = data.DataReader(name='^DJI', data_source='stooq', start=start_date, end=end_date)[['Close']]

In [ ]:
# Let us use the stocks dataset from earlier
df = aapl.join(qqq, lsuffix='_l', rsuffix='_r').join(bbby)
df.head()

In [ ]:
# reset index to make Date a column
df = aapl.join(qqq, lsuffix='_l', rsuffix='_r').join(bbby).reset_index()
# Wide Format
df.head()

In [ ]:
# Convert Wide format To Long Format using melt
df_long = pd.melt(df, ['Date'])
df_long.sample(10)

- *Pivoting "Long" to "Wide" Format*
- Long or *stacked* format is a representation of storing data where each row in the table represents a single observations
- In such a format, one or more column values in a row might repeat in another row
- Alternatively such a format can be transformed into a Wide or *unstacked* format where a DataFrame can contain one column per distinct item value
- The *pivot* instance method is used to perform this operation
- Its first two argument represents the row and column indices of the pivoted table. The third argument is a optional and selects the value column to fill the DataFrame
- Omitting the third argument results in a DataFrame with hierarchical columns
- The *pivot* operation is equivalent to creating a hierarchical index using *set_index* followed by call to *unstack*

In [ ]:
# Convert Long to stacked format using pivot (reshape back to wide format)
df_long.pivot(index='Date', columns='variable', values='value').head()

In [ ]:
df_long.pivot(index='Date', columns='variable').head() # emitting the last argument results in a DataFrame with hierarchical columns

In [ ]:
# use set_index followed by unstack to achieve the same result
df_long.set_index(['Date', 'variable']).unstack('variable').head()